# ColabFold Batch Prediction - Laminarinases
## 62 Medium/Large Sequences (400-2435 aa)

This notebook will:
1. Install ColabFold
2. Upload your FASTA file
3. Run batch predictions
4. Generate summary with confidence scores
5. Download all results as ZIP

**Estimated time:** 30-90 minutes (depends on sequence sizes and queue)

## Step 1: Install ColabFold
Run this cell to install ColabFold with AlphaFold2

In [1]:
import os
import sys

# Verify we're in Colab
try:
    from google.colab import files
    IN_COLAB = True
    print("✅ Running in Google Colab")
except:
    IN_COLAB = False
    print("⚠️  Not in Colab - this notebook is designed for Google Colab")
    print("Please open this notebook in Colab: https://colab.research.google.com/")

# Install ColabFold
if IN_COLAB:
    print("\n📦 Installing ColabFold...")
    !pip install -q "colabfold[alphafold] @ git+https://github.com/sokrypton/ColabFold" 2>&1 | tail -20
    print("✅ Installation complete!")

✅ Running in Google Colab

📦 Installing ColabFold...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.4/248.4 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 75.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 373.8/373.8 kB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 63.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.0/259.0 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.1/105.1 MB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 84.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.7/76.7 kB 5.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.19.0 requires tensorboard~=2.19.0, but you have tensorboard 2.20.0 which is incompatible.
flax 0.10.7 requires jax>=0.6.0, but yo

## Step 2: Upload FASTA File

Click "Choose Files" and select `laminarinases_medium_large_61.fasta`

In [6]:
print("="*80)
print("UPLOAD FASTA FILE")
print("="*80)

# Upload file using Colab's upload widget
print("\nClick 'Choose Files' and select: laminarinases_medium_large_61.fasta")
print("Contains: 62 sequences (400-2435 aa)")
print()

uploaded = files.upload()
fasta_file = list(uploaded.keys())[0]
print(f"\n✅ Uploaded: {fasta_file}")

# Verify sequences
from Bio import SeqIO
records = list(SeqIO.parse(fasta_file, "fasta"))
print(f"\n✅ Loaded {len(records)} sequences")
print(f"   Size range: {min(len(r.seq) for r in records)} - {max(len(r.seq) for r in records)} aa")
print(f"   Mean size: {sum(len(r.seq) for r in records) // len(records)} aa")

UPLOAD YOUR FASTA FILE

Expected file: laminarinases_medium_large_61.fasta
Contains: 62 sequences (400-2435 aa)



KeyboardInterrupt: 

## Step 3: Configure Prediction Settings

In [7]:
# Configuration
jobname = "laminarinases_batch"
output_dir = f"/content/{jobname}"
num_models = 1  # Set to 1 for speed, 5 for higher accuracy
num_recycles = 3  # Default value
use_amber = True  # GPU-accelerated relaxation
use_templates = False  # No homolog templates

print("="*80)
print("CONFIGURATION SUMMARY")
print("="*80)
print(f"Job name:           {jobname}")
print(f"Output directory:   {output_dir}")
print(f"Models per seq:     {num_models} (1=fast, 5=accurate)")
print(f"Recycling cycles:   {num_recycles}")
print(f"AMBER relaxation:   {use_amber}")
print(f"Templates:          {use_templates}")

# Check GPU
import subprocess
try:
    gpu_info = subprocess.check_output('nvidia-smi --query-gpu=name --format=csv,noheader', shell=True).decode().strip()
    print(f"GPU available:      {gpu_info}")
except:
    print("GPU available:      None detected (warning)")

print("="*80)
print(f"\n⏳ Ready to predict {len(records)} sequences")
print("   Estimated time: 30-90 minutes depending on sizes")
print("="*80)

CONFIGURATION SUMMARY
Job name:           laminarinases_batch
Output directory:   /content/laminarinases_batch
Models per seq:     1 (1=fast, 5=accurate)
Recycling cycles:   3
AMBER relaxation:   True
Templates:          False
GPU available:      None detected (warning)


NameError: name 'records' is not defined

## Step 4: Run Batch Prediction

**This step will take 30-90 minutes. Do not disconnect!**

In [ ]:
import os
os.makedirs(output_dir, exist_ok=True)

print("="*80)
print("RUNNING BATCH PREDICTION")
print("="*80)
print(f"Input file:  {fasta_file}")
print(f"Output dir:  {output_dir}")
print(f"Sequences:   {len(records)}")
print("\n⏳ Processing... This will take a while!")
print("="*80)

# Run ColabFold
cmd = f"colabfold_batch {fasta_file} {output_dir} --num-models {num_models} --num-recycle {num_recycles}"
if use_amber:
    cmd += " --use-gpu-relax"

print(f"\nCommand: {cmd}\n")
os.system(cmd)

## Step 5: Review Results

In [ ]:
import glob
import json

print("="*80)
print("RESULTS SUMMARY")
print("="*80)

# Find output files
pdb_files = glob.glob(f"{output_dir}/*_rank_001_*.pdb")
json_files = glob.glob(f"{output_dir}/*.json")

print(f"PDB structures:     {len(pdb_files)}")
print(f"Confidence files:   {len(json_files)}")

# Load confidence scores
if json_files:
    predictions = []
    for json_file in json_files:
        try:
            with open(json_file) as f:
                data = json.load(f)
                name = os.path.basename(json_file).replace('.json', '')
                plddt = data.get('plddt', data.get('mean_plddt', 0))
                predictions.append((name, plddt))
        except:
            pass
    
    if predictions:
        predictions.sort(key=lambda x: x[1], reverse=True)
        
        scores = [p[1] for p in predictions]
        print(f"\npLDDT range:        {min(scores):.2f} - {max(scores):.2f}")
        print(f"Mean pLDDT:         {sum(scores)/len(scores):.2f}")
        
        # Quality distribution
        high = sum(1 for s in scores if s > 90)
        good = sum(1 for s in scores if 70 <= s <= 90)
        low = sum(1 for s in scores if s < 70)
        
        print(f"\nQuality distribution:")
        print(f"  High (>90):       {high} structures ({100*high/len(scores):.1f}%)")
        print(f"  Good (70-90):     {good} structures ({100*good/len(scores):.1f}%)")
        print(f"  Low (<70):        {low} structures ({100*low/len(scores):.1f}%)")
        
        print(f"\n📊 Top 10 by confidence:")
        for i, (name, score) in enumerate(predictions[:10], 1):
            print(f"  {i:2d}. {name:40s} pLDDT: {score:.2f}")

print("\n" + "="*80)
print("✅ Predictions complete!")
print("="*80)

## Step 6: Download Results

In [ ]:
import shutil

print("="*80)
print("PREPARING DOWNLOAD")
print("="*80)

# Create ZIP file
zip_name = f"{jobname}_results"
print(f"\n📦 Creating ZIP archive: {zip_name}.zip")
print(f"   Source: {output_dir}")
print(f"   Size: {sum(os.path.getsize(f) for f in glob.glob(f'{output_dir}/*')) / 1e6:.1f} MB")

shutil.make_archive(zip_name, 'zip', output_dir)
print(f"✅ Created: {zip_name}.zip")

# Download
print(f"\n📥 Downloading file...")
files.download(f"{zip_name}.zip")
print(f"✅ Download started!")

print("\n" + "="*80)
print("NEXT STEPS")
print("="*80)
print("1. Extract the ZIP file on your local machine")
print(f"   {zip_name}/")
print("")
print("2. Run the processing script:")
print("   python process_colabfold_results.py")
print("")
print("3. This will:")
print("   - Rename files to standard format")
print("   - Copy to predicted_structures_alphafold/")
print("   - Generate summary report")
print("")
print("4. Combine with existing predictions:")
print("   - 19 ESMFold (already have)")
print("   - 62 AlphaFold (from this notebook)")
print("   - Total: 81 ML-predicted structures")
print("")
print("5. Run batch MD:")
print("   python run_batch_md_sequential.py")
print("="*80)